# Stage 2 Detection — Inference Benchmark

Runs `model.val()` on the stratified `val.txt` split for each stage-2 model (n/s/m/l/x) and plots **mAP@50 vs inference time**.

> Val set is `val.txt` (stratified split) — the same split used during training — to avoid data leakage.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

from pathlib import Path

RUNS_DIR    = Path("runs/detect/stage2_detect_target")   # relative to notebook
SIZES       = ["n", "s", "m", "l", "x"]

# yaml points to val.txt (stratified split)
EVAL_YAML   = "2025_pollen_stratified.yaml"

IMGSZ       = 640
BATCH       = 16
DEVICE      = "0"
SINGLE_CLS  = True    # must match training config

PLOTS_DIR   = Path("../plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from ultralytics import YOLO

In [3]:
# ── Run val for each model ────────────────────────────────────────────────────
records = []

for size in SIZES:
    weights = RUNS_DIR / f"yolo26{size}_detect_target" / "weights" / "best.pt"
    assert weights.exists(), f"Weights not found: {weights}"

    print(f"\n{'='*55}")
    print(f" Evaluating  yolo26{size}  ({weights})")
    print(f"{'='*55}")

    model   = YOLO(str(weights))
    results = model.val(
        data       = EVAL_YAML,
        imgsz      = IMGSZ,
        batch      = BATCH,
        device     = DEVICE,
        single_cls = SINGLE_CLS,
        verbose    = True,
    )

    map50          = float(results.box.map50)
    infer_ms       = float(results.speed["inference"])   # ms / image
    preproc_ms     = float(results.speed["preprocess"])
    postproc_ms    = float(results.speed["postprocess"])

    rec = {
        "model"         : f"yolo26{size}",
        "size"          : size,
        "mAP50"         : round(map50, 4),
        "infer_ms"      : round(infer_ms, 3),
        "preproc_ms"    : round(preproc_ms, 3),
        "postproc_ms"   : round(postproc_ms, 3),
        "total_ms"      : round(preproc_ms + infer_ms + postproc_ms, 3),
    }
    records.append(rec)
    print(f"  mAP@50={map50:.4f}   infer={infer_ms:.2f} ms/img")

df = pd.DataFrame(records).set_index("model")
print("\nSummary:")
print(df[["mAP50", "infer_ms", "total_ms"]].to_string())


 Evaluating  yolo26n  (runs/detect/stage2_detect_target/yolo26n_detect_target/weights/best.pt)
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 4.6±1.6 ms, read: 35.3±3.9 MB/s, size: 3366.7 KB)
val: Scanning /auto/vestec1-elixir/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/selection4class_training_27052024.cache... 592 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 592/592 75.2Mit/s 0.0s


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 4.5it/s 8.2s<0.1s
                   all        592       7128      0.947      0.964      0.984      0.946
Speed: 0.7ms preprocess, 2.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /auto/vestec1-elixir/home/mamedove/Scripts/experimental/YOLO26/runs/detect/val
  mAP@50=0.9845   infer=2.33 ms/img

 Evaluating  yolo26s  (runs/detect/stage2_detect_target/yolo26s_detect_target/weights/best.pt)
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6570.7±1233.8 MB/s, size: 3411.2 KB)
val: Scanning /auto/vestec1-elixir/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/selection4class_training_27052024.cache... 592 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 592

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 5.8it/s 6.4s0.1ss
                   all        592       7128       0.95      0.972      0.986      0.956
Speed: 0.7ms preprocess, 2.6ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /auto/vestec1-elixir/home/mamedove/Scripts/experimental/YOLO26/runs/detect/val2
  mAP@50=0.9861   infer=2.58 ms/img

 Evaluating  yolo26m  (runs/detect/stage2_detect_target/yolo26m_detect_target/weights/best.pt)
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5961.8±1305.1 MB/s, size: 3140.8 KB)
val: Scanning /auto/vestec1-elixir/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/selection4class_training_27052024.cache... 592 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 4.8it/s 7.7s0.1s
                   all        592       7128      0.945      0.976      0.987       0.96
Speed: 0.5ms preprocess, 4.9ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /auto/vestec1-elixir/home/mamedove/Scripts/experimental/YOLO26/runs/detect/val3
  mAP@50=0.9866   infer=4.86 ms/img

 Evaluating  yolo26l  (runs/detect/stage2_detect_target/yolo26l_detect_target/weights/best.pt)
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
YOLO26l summary (fused): 190 layers, 24,746,511 parameters, 0 gradients, 86.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4248.9±1864.8 MB/s, size: 3240.8 KB)
val: Scanning /auto/vestec1-elixir/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/selection4class_training_27052024.cache... 592 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 59

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 4.4it/s 8.4s0.2s
                   all        592       7128      0.949      0.974      0.986       0.96
Speed: 0.8ms preprocess, 6.2ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /auto/vestec1-elixir/home/mamedove/Scripts/experimental/YOLO26/runs/detect/val4
  mAP@50=0.9864   infer=6.22 ms/img

 Evaluating  yolo26x  (runs/detect/stage2_detect_target/yolo26x_detect_target/weights/best.pt)
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
YOLO26x summary (fused): 190 layers, 55,634,703 parameters, 0 gradients, 193.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6091.6±747.9 MB/s, size: 3043.4 KB)
val: Scanning /auto/vestec1-elixir/home/mamedove/00_Data/2025_Pollen/Multi_class/multi-class-set/selection4class_training_27052024.cache... 592 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 59

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 3.7it/s 10.1s.2s
                   all        592       7128      0.952      0.973      0.987      0.961
Speed: 0.8ms preprocess, 10.1ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /auto/vestec1-elixir/home/mamedove/Scripts/experimental/YOLO26/runs/detect/val5
  mAP@50=0.9872   infer=10.09 ms/img

Summary:
          mAP50  infer_ms  total_ms
model                              
yolo26n  0.9845     2.329     3.156
yolo26s  0.9861     2.582     3.428
yolo26m  0.9866     4.864     5.484
yolo26l  0.9864     6.219     7.151
yolo26x  0.9872    10.085    10.997


In [4]:
# ── Save results to CSV ───────────────────────────────────────────────────────
csv_path = PLOTS_DIR / "stage2_inference_benchmark.csv"
df.to_csv(csv_path)
print(f"Saved → {csv_path}")

Saved → ../plots/stage2_inference_benchmark.csv


In [ ]:
# ── Plot: mAP@50 vs inference time ────────────────────────────────────────────
SIZE_ORDER = {"n": 0, "s": 1, "m": 2, "l": 3, "x": 4}
COLORS     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

fig, ax = plt.subplots(figsize=(7, 5))

for i, row in df.reset_index().iterrows():
    ax.scatter(
        row["infer_ms"], row["mAP50"],
        s=120, color=COLORS[i], zorder=3, label=row["model"],
    )
    ax.annotate(
        row["model"],
        xy=(row["infer_ms"], row["mAP50"]),
        xytext=(6, 4), textcoords="offset points",
        fontsize=9,
        path_effects=[pe.withStroke(linewidth=2, foreground="white")],
    )

# Connect points in size order (n → s → m → l → x)
df_sorted = df.reset_index().sort_values("infer_ms")
ax.plot(df_sorted["infer_ms"], df_sorted["mAP50"],
        color="gray", linewidth=1, linestyle="--", zorder=2, alpha=0.6)

ax.set_xlabel("Inference time (ms / image)", fontsize=11)
ax.set_ylabel("mAP@50", fontsize=11)
ax.set_title("Stage 2 Detection — mAP@50 vs Inference Time\n"
             "(val.txt stratified, single_cls=True, imgsz=640)", fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plot_path = PLOTS_DIR / "stage2_map50_vs_infer_time.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved → {plot_path}")